# Titanic Survival Prediction

ECE 3100: Introduction to Data Science, University of Michigan

**Goal:** predict whether a Titanic passenger survived (`1`) or did not (`0`) from passenger characteristics.
This is a supervised binary classification problem solved with a scikit-learn logistic regression pipeline.

**Workflow:** load data, explore, engineer features, build a preprocessing + model `Pipeline`, evaluate with a
stratified train/test split and 5-fold stratified cross-validation, then interpret the confusion matrix.

> Cleanup note: this version of the notebook keeps the original modeling choices and results. The only change to the
> core workflow is that all imputation and scaling now happens *inside* the pipeline, so nothing is fit on test data
> before the split. The final section contains optional extensions that were added after the original submission.

## 1. Setup

In [11]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 2. Load the data

The dataset is the Stanford CS109 version of the Titanic passenger list (887 passengers).

In [12]:
URL = "https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv"

df = pd.read_csv(URL)
print("Shape:", df.shape)
df.head()

Shape: (887, 8)


,Survived,Pclass,Name,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


## 3. Exploratory data analysis

In [13]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Overall survival rate:", round(df["Survived"].mean(), 3))

Missing values per column:
Survived                   0
Pclass                     0
Name                       0
Sex                        0
Age                        0
Siblings/Spouses Aboard    0
Parents/Children Aboard    0
Fare                       0
dtype: int64

Overall survival rate: 0.386


## 4. Feature engineering

- `Sex` is encoded as 0 (male) / 1 (female).
- `AgeGroup` bins `Age` into five categories. Bin edges: 0-12 Child, 13-19 Teenager, 20-35 Young Adult,
  36-59 Adult, 60+ Senior.

Imputation and scaling are **not** done here. They happen inside the pipeline in section 5, so they are learned from
training data only.

In [14]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

age_bins = [0, 12, 19, 35, 59, np.inf]
age_labels = ["Child", "Teenager", "Young Adult", "Adult", "Senior"]

df["AgeGroup"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, include_lowest=True)

df["AgeGroup"].value_counts()

,count
AgeGroup,
Young Adult,426
Adult,231
Teenager,120
Child,79
Senior,31


In [15]:
# Survival rate by key groups (descriptive only; not used by the model)
for col in ["Sex", "Pclass", "AgeGroup"]:
    print(df.groupby(col, observed=True)["Survived"].agg(["mean", "count"]).round(3))
    print()

      mean  count
Sex              
0    0.190    573
1    0.742    314

         mean  count
Pclass              
1       0.630    216
2       0.473    184
3       0.244    487

              mean  count
AgeGroup                 
Child        0.544     79
Teenager     0.383    120
Young Adult  0.366    426
Adult        0.390    231
Senior       0.226     31



## 5. Preprocessing and model pipeline

| Feature type | Columns | Steps |
|---|---|---|
| Numeric | `Pclass`, `Age`, `Fare`, `Sex` | median imputation, then standard scaling |
| Categorical | `AgeGroup` | most-frequent imputation, then one-hot encoding |

The `ColumnTransformer` is wrapped together with `LogisticRegression` in a single `Pipeline`, so every step is
fit on training data only, including inside each cross-validation fold.

In [16]:
numeric_features = ["Pclass", "Age", "Fare", "Sex"]
categorical_features = ["AgeGroup"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

## 6. Train/test split and training

In [17]:
features = numeric_features + categorical_features
X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

Training set: (709, 5)
Testing set: (178, 5)


## 7. Evaluation

In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Reported metric: 5-fold stratified CV over the full dataset
cv_scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy")
mean_cv_acc = cv_scores.mean()

print("Cross-validation scores:", cv_scores.round(4))
print("Mean cross-validation accuracy:", round(mean_cv_acc, 4))
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Cross-validation scores: [0.809  0.7753 0.7684 0.8475 0.7966]
Mean cross-validation accuracy: 0.7993
Test accuracy: 0.7809

Confusion matrix (rows = actual, columns = predicted):
[[89 20]
 [19 50]]

              precision    recall  f1-score   support

           0       0.82      0.82      0.82       109
           1       0.71      0.72      0.72        69

    accuracy                           0.78       178
   macro avg       0.77      0.77      0.77       178
weighted avg       0.78      0.78      0.78       178



In [19]:
# Stricter check: cross-validate on the training set only, so the test set plays no part in model assessment
cv_train_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")
print("Mean CV accuracy (training set only):", round(cv_train_scores.mean(), 4))

Mean CV accuracy (training set only): 0.804


### How to read the confusion matrix

Using scikit-learn's convention (rows = actual, columns = predicted; class 0 = did not survive, class 1 = survived):

| | Predicted: did not survive | Predicted: survived |
|---|---:|---:|
| **Actual: did not survive** | 89 (true negatives) | 20 (false positives) |
| **Actual: survived** | 19 (false negatives) | 50 (true positives) |

- **Accuracy** = (89 + 50) / 178 = 78.09%
- **Precision (survived)** = 50 / (50 + 20) = 71.4%: of passengers predicted to survive, how many did.
- **Recall (survived)** = 50 / (50 + 19) = 72.5%: of passengers who survived, how many the model caught.
- **Specificity** = 89 / (89 + 20) = 81.7%: of passengers who died, how many the model got right.

## 8. Conclusion

The logistic regression model reached a mean 5-fold cross-validation accuracy of about 79.93% and a test accuracy of
about 78.09%. Because the two numbers are close, performance looks consistent on unseen data.

The model predicts non-survivors better (F1 0.82) than survivors (F1 0.72). Keeping imputation, scaling and encoding
inside a `Pipeline` with a `ColumnTransformer` keeps preprocessing consistent and prevents information from the
validation or test data from leaking into training. Stratified cross-validation keeps the survival rate similar
across folds.

**Limitations**
- The test set has only 178 passengers, so the test accuracy has a margin of error of roughly 6 percentage points.
- Hyperparameters were left at scikit-learn defaults and no other algorithms were compared in the original work.
- `Age` and `AgeGroup` are both model inputs, so they carry overlapping information.
- Accuracy is the headline metric; the classes are mildly imbalanced (about 61% did not survive).

## 9. Optional extensions (added after the original submission)

These cells are **not** part of the original project results. They put the logistic regression score in context with a
baseline, a feature ablation and one alternative model, all scored with the same 5-fold cross-validation.

In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

def make_pipeline(classifier, numeric, categorical):
    transformers = [("num", numeric_pipeline, numeric)]
    if categorical:
        transformers.append(("cat", categorical_pipeline, categorical))
    return Pipeline([
        ("preprocessor", ColumnTransformer(transformers)),
        ("classifier", classifier),
    ])

candidates = {
    "Majority-class baseline": make_pipeline(DummyClassifier(strategy="most_frequent"), numeric_features, categorical_features),
    "Logistic regression (original)": model,
    "Logistic regression without AgeGroup": make_pipeline(LogisticRegression(max_iter=1000), numeric_features, []),
    "Random forest": make_pipeline(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE), numeric_features, categorical_features),
}

rows = []
for name, estimator in candidates.items():
    scores = cross_val_score(estimator, X[features], y, cv=skf, scoring="accuracy")
    rows.append({"model": name, "mean_cv_accuracy": round(scores.mean(), 4), "std": round(scores.std(), 4)})

pd.DataFrame(rows)

,model,mean_cv_accuracy,std
0,Majority-class baseline,0.6144,0.0017
1,Logistic regression (original),0.7993,0.0281
2,Logistic regression without AgeGroup,0.7892,0.0357
3,Random forest,0.8219,0.0181
